### Process DimUser

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
import os, sys
sys.path = [p for p in sys.path if 'spotify_bundle' not in p]
project_path = os.path.abspath(os.path.join(
    '/Workspace/Users/snavaratna23@gmail.com/spotify_bundle'
))
sys.path.insert(0, project_path)

print("Added path:", project_path)
print("Utils exists:", os.path.exists(os.path.join(project_path, 'utils')))
print("Init exists:", os.path.exists(os.path.join(project_path, 'utils', '__init__.py')))
print("File exists:", os.path.exists(os.path.join(project_path, 'utils', 'transformations.py')))

from utils.transformations import resuable_bundle

In [0]:
df_user = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format", "parquet")\
              .option("cloudFiles.schemaLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimUser/schema")\
              .load("abfss://bronze@azurestoragegouri.dfs.core.windows.net/DimUser")

In [0]:
df_user = df_user.withColumn('user_name', upper(col('user_name')))

In [0]:
df_user_obj = resuable_bundle()

df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
display(df_user, checkpointLocation = "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimUser/checkpoint_display1")

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimUser/checkpoint_write")\
    .trigger(once=True)\
    .option("path", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_catalog.silver.DimUser")

### Process DimArtist

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
              .option("cloudFiles.format", "parquet")\
              .option("cloudFiles.schemaLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimArtist/schema")\
              .load("abfss://bronze@azurestoragegouri.dfs.core.windows.net/DimArtist")

In [0]:
display(df_artist, checkpointLocation = "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimArtist/checkpoint_display")

In [0]:
df_artist = df_user_obj.dropColumns(df_artist, ['_rescued_data'])

In [0]:
df_artist.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimArtist/checkpoint_write")\
    .trigger(once=True)\
    .option("path", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_catalog.silver.DimArtist")

### DimTrack

In [0]:
df_track = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet").option("cloudFiles.schemaLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimTrack/schema")\
            .load("abfss://bronze@azurestoragegouri.dfs.core.windows.net/DimTrack")

In [0]:
df_track = df_track.withColumn("durationFlag", when(col('duration_sec')< 150, "low")\
                                               .when(col('duration_sec') < 300, "medium")\
                                                .otherwise("high"))

In [0]:
df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), '-', ' '))

In [0]:
df_track = resuable_bundle().dropColumns(df_track, ['_rescued_data'])

In [0]:
display(df_track, checkpointLocation = "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimTrack/checkpoint_display1")

In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimTrack/checkpoint_write")\
    .trigger(once=True)\
    .option("path", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_catalog.silver.DimTrack")

### DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimDate/schema")\
            .option("schemaEvolutionMode", "addNewColumns")\
            .load("abfss://bronze@azurestoragegouri.dfs.core.windows.net/DimDate")

In [0]:
display(df_date, checkpointLocation = "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimDate/checkpoint_display")

In [0]:
df_date = resuable_bundle().dropColumns(df_date, ['_rescued_data'])

In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimDate/checkpoint_write")\
    .trigger(once=True)\
    .option("path", "abfss://silver@azurestoragegouri.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_catalog.silver.DimDate")

### FactStream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format", "parquet")\
            .option("cloudFiles.schemaLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/FactStream/schema")\
            .option("schemaEvolutionMode", "addNewColumns")\
            .load("abfss://bronze@azurestoragegouri.dfs.core.windows.net/FactStream")

In [0]:
df_fact = resuable_bundle().dropColumns(df_fact, ['_rescued_data'])

In [0]:
df_fact.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@azurestoragegouri.dfs.core.windows.net/FactStream/checkpoint_write")\
    .trigger(once=True)\
    .option("path", "abfss://silver@azurestoragegouri.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_catalog.silver.FactStream")

In [0]:
%sql
select * from spotify_catalog.gold.dimtrack
where track_id IN (46,5)

In [0]:
%sql
select * from spotify_catalog.gold.dimuser
where `__END_AT` IS NOT NULL